# Exploring Mamba's Selective State Space (S6)
**ArXivist-generated exploratory notebook**
Paper: https://arxiv.org/abs/2312.00752v2

This notebook visualizes the time-varying parameters ($B_t, C_t, \Delta_t$) inside the Selective SSM to see how Mamba achieves content-based reasoning.

*Note: This uses an untrained model for demonstration. For meaningful representations, load a pretrained checkpoint.*

In [ ]:
import sys, torch
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F
from mamba.models.blocks import SelectiveSSM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### Visualizing Time-Varying Parameters

In standard SSMs, $B$, $C$, and $\Delta$ are constant across time. In Mamba, they are generated dynamically from the input $x_t$.

In [ ]:
try:
    # Create a dummy model
    d_model = 64
    d_state = 16
    dt_rank = 4
    ssm = SelectiveSSM(d_model=d_model, d_state=d_state, dt_rank=dt_rank).to(device)
    
    # Generate synthetic input (1 sample, sequence length 50)
    seq_len = 50
    x = torch.randn(1, seq_len, d_model).to(device)
    
    # Extract projections from the model manually to visualize them
    x_proj = ssm.x_proj(x)
    dt, B_mat, C = torch.split(x_proj, [dt_rank, d_state, d_state], dim=-1)
    dt = ssm.dt_proj(dt)
    dt = F.softplus(dt)
    
    dt_np = dt[0].detach().cpu().numpy()
    B_np = B_mat[0].detach().cpu().numpy()
    C_np = C[0].detach().cpu().numpy()
    
    fig, axes = plt.subplots(3, 1, figsize=(10, 12))
    
    sns.heatmap(dt_np.T, ax=axes[0], cmap="viridis")
    axes[0].set_title("$\Delta_t$ (Step Size) over Time")
    axes[0].set_xlabel("Time Step (t)")
    axes[0].set_ylabel("Dimension")
    
    sns.heatmap(B_np.T, ax=axes[1], cmap="magma")
    axes[1].set_title("$B_t$ (Input Matrix) over Time")
    axes[1].set_xlabel("Time Step (t)")
    axes[1].set_ylabel("State Dimension")
    
    sns.heatmap(C_np.T, ax=axes[2], cmap="plasma")
    axes[2].set_title("$C_t$ (Output Matrix) over Time")
    axes[2].set_xlabel("Time Step (t)")
    axes[2].set_ylabel("State Dimension")
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Error generating visualizations: {e}")